# 01 — Thu thập dữ liệu tử vong Việt Nam

**Mục tiêu:** tải và kiểm tra dữ liệu từ UN WPP (nguồn chính), đối chiếu GSO.

**Checklist:**
- [ ] Tải bảng sống tuổi đơn UN WPP (1950–nay) vào `data/raw/`
- [ ] Tải bảng sống rút gọn GSO (2009, 2019) vào `data/external/`
- [ ] Kiểm tra tính liên tục theo năm, missing values
- [ ] Chạy `python -m src.data.make_dataset` để dựng Dxt, Ext
- [ ] Ghi chú nguồn + ngày tải vào `data/external/SOURCES.md`

## Tải dữ liệu (thủ công)

UN WPP không cho tải qua API đơn giản cho toàn bộ chuỗi 1955–2023, nên tải thủ công 3 file "Single age life table estimates" tại https://population.un.org/wpp/ (mục *Life Tables*):

| Giới tính | Tên file (đặt vào `data/raw/`) |
|---|---|
| Cả hai giới | `WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx` |
| Nam | `WPP2024_MORT_F06_2_SINGLE_AGE_LIFE_TABLE_ESTIMATES_MALE.xlsx` |
| Nữ | `WPP2024_MORT_F06_3_SINGLE_AGE_LIFE_TABLE_ESTIMATES_FEMALE.xlsx` |

Ghi lại ngày tải vào `data/external/SOURCES.md`. Bảng sống GSO (đối chiếu, không dùng để fit) số hoá thủ công từ báo cáo TĐT thành CSV với cột tối thiểu `age`, `sex`, `mx` hoặc `qx`, đặt vào `data/external/`.

In [ ]:
from src.config import DATA_RAW, DATA_PROCESSED, load_params
from src.data.make_dataset import WPP_FILES, main as build_dataset

cfg = load_params()
missing = [f for f in WPP_FILES.values() if not (DATA_RAW / f).exists()]
if missing:
    print("Còn thiếu trong data/raw/:")
    for f in missing:
        print(" -", f)
else:
    print("Đã có đủ 3 file UN WPP, dựng Dxt/Ext...")
    build_dataset()

## Kiểm tra nhanh sau khi dựng ma trận

Chạy sau khi cell trên đã build thành công (3 file `Dxt_*.csv` / `Ext_*.csv` xuất hiện trong `data/processed/`).

In [ ]:
import pandas as pd

Dxt = pd.read_csv(DATA_PROCESSED / "Dxt_total.csv", index_col=0)
Dxt.columns = Dxt.columns.astype(int)
Ext = pd.read_csv(DATA_PROCESSED / "Ext_total.csv", index_col=0)
Ext.columns = Ext.columns.astype(int)

print("Dxt:", Dxt.shape, "| tuổi", Dxt.index.min(), "-", Dxt.index.max(),
      "| năm", Dxt.columns.min(), "-", Dxt.columns.max())
print("Số ô thiếu:", Dxt.isna().sum().sum())
mx = Dxt / Ext
print("mx min/max:", mx.values.min(), mx.values.max())
mx.head()

In [ ]:
from src.visualization.plots import plot_log_mx_by_age

plot_log_mx_by_age(mx, years=[1960, 1980, 2000, 2020])